# 🚀 Automated System & Hardware Check for NeMo / Automodel

This notebook automatically verifies whether your host system meets all requirements (disk space, NVIDIA drivers, Docker & GPU passthrough) to launch the containers.

**Workflow:**
Simply execute the cells sequentially. If a check fails, the notebook will clearly indicate what needs to be resolved.

In [1]:
# Import required libraries
import subprocess
import shutil
import sys
import os
import socket
from pathlib import Path

class HardwareCheckError(Exception):
    """Custom exception for hardware/environment errors."""
    pass

print("✅ Python environment ready.")

✅ Python environment ready.


### Step 1: Check Disk Space
Verifies whether sufficient disk space (e.g., for container images and model caches) is available.

In [2]:
def check_disk_space(required_gb=250):
    print("🔍 [1/4] Checking available disk space...")
    total, used, free = shutil.disk_usage(".")
    free_gb = free / (1024 ** 3)
    print(f"   --> Available disk space: {free_gb:.2f} GB")
    
    if free_gb < required_gb:
        raise HardwareCheckError(
            f"Insufficient disk space! Required: at least {required_gb} GB, Available: {free_gb:.2f} GB.\n"
            f"Tip: Large language models and container images require significant storage."
        )
    print("   ✅ Disk capacity sufficient!\n")

try:
    check_disk_space(required_gb=250)
except HardwareCheckError as e:
    print(f"❌ ERROR: {e}")

🔍 [1/4] Checking available disk space...
   --> Available disk space: 165.65 GB
❌ ERROR: Insufficient disk space! Required: at least 250 GB, Available: 165.65 GB.
Tip: Large language models and container images require significant storage.


### Step 2: Check NVIDIA Drivers & GPU Availability
Verifies whether `nvidia-smi` responds on the host and checks how much VRAM is available.

In [3]:
def check_nvidia_smi():
    print("🔍 [2/4] Checking host NVIDIA drivers & GPU availability...")
    
    # 1. Display full standard output of nvidia-smi (Optional)
    try:
        raw_result = subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True)
        print(raw_result.stdout)
    except Exception as e:
        print(f"Could not load full nvidia-smi table: {e}")

    # 2. Existing logic for VRAM check
    try:
        result = subprocess.run(
            ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.free", "--format=csv,noheader,nounits"],
            capture_output=True,
            text=True,
            check=True
        )
        gpus = result.stdout.strip().split("\n")
        print(f"  --> Detected GPUs ({len(gpus)}):")
        
        total_free_vram = 0
        for gpu in gpus:
            idx, name, mem_total, mem_free = [x.strip() for x in gpu.split(",")]
            free_vram = int(mem_free) / 1024
            total_free_vram += free_vram
            print(f"   • GPU {idx}: {name} | Free VRAM: {free_vram:.2f} GB / {int(mem_total)/1024:.2f} GB")
            
        print(f"  --> Total free VRAM across all GPUs: {total_free_vram:.2f} GB")
        if total_free_vram < 40:
            print("   ⚠️ WARNING: Free VRAM might be tight for large LLMs!")
        print("   ✅ Host GPU drivers reachable!\n")
        
    except FileNotFoundError:
        raise HardwareCheckError("'nvidia-smi' was not found. Please install the NVIDIA drivers on the host.")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(f"Error executing nvidia-smi: {e.stderr}")

try:
    check_nvidia_smi()
except HardwareCheckError as e:
    print(f"❌ ERROR: {e}\n👉 Please resolve the driver issue before continuing.")

🔍 [2/4] Checking host NVIDIA drivers & GPU availability...
Mon Nov 16 20:50:18 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 595.112.07             Driver Version: 595.112.07     CUDA Version: 13.2     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L40S                    On  |   00000000:30:00.0 Off |                    0 |
| N/A   76C    P0            217W /  350W |   41643MiB /  46068MiB |    100%      Default |
|                                         |                        |             

### Step 3: Check Docker Installation & Daemon
Checks whether the Docker service is running and the user has the required permissions.

In [4]:
def check_docker_installation():
    print("🔍 [3/4] Checking Docker installation & daemon status...")
    try:
        subprocess.run(["docker", "info"], capture_output=True, text=True, check=True)
        print("   ✅ Docker is installed and the daemon is running!\n")
    except FileNotFoundError:
        raise HardwareCheckError("Docker CLI not found. Is Docker installed on the system?")
    except subprocess.CalledProcessError:
        raise HardwareCheckError("Docker daemon is not running or the current user lacks permissions (sudo / docker group).")

try:
    check_docker_installation()
except HardwareCheckError as e:
    print(f"❌ ERROR: {e}\n👉 Start the Docker service or check your user permissions.")

🔍 [3/4] Checking Docker installation & daemon status...
❌ ERROR: Docker CLI not found. Is Docker installed on the system?
👉 Start the Docker service or check your user permissions.


### Step 4: Check NVIDIA Container Toolkit (GPU Passthrough)
Tests whether a Docker container can successfully access the GPU.

In [5]:
def check_nvidia_container_toolkit():
    print("🔍 [4/4] Checking NVIDIA Container Toolkit (GPU passthrough in Docker)...")
    cmd = [
        "docker", "run", "--rm", "--gpus", "all",
        "nvidia/cuda:12.0.0-base-ubuntu22.04",
        "nvidia-smi"
    ]
    try:
        subprocess.run(cmd, capture_output=True, text=True, check=True)
        print("   ✅ GPU passthrough to Docker containers works properly!\n")
    except FileNotFoundError:
        raise HardwareCheckError("Docker CLI not found. Please install Docker first.")
    except subprocess.CalledProcessError as e:
        raise HardwareCheckError(
            f"Docker cannot access the GPUs! Error:\n{e.stderr}\n"
            "Tip: Make sure 'nvidia-container-toolkit' is installed and Docker has been restarted."
        )

try:
    check_nvidia_container_toolkit()
except HardwareCheckError as e:
    print(f"❌ ERROR: {e}\n👉 Install the NVIDIA Container Toolkit (see README).")

🔍 [4/4] Checking NVIDIA Container Toolkit (GPU passthrough in Docker)...
❌ ERROR: Docker CLI not found. Please install Docker first.
👉 Install the NVIDIA Container Toolkit (see README).


### Step 5: Prepare Project Directories & Environment
Once all checks above pass, we set up the folders and optionally configure the `.env` tokens.

In [ ]:
from pathlib import Path
import os
from dotenv import load_dotenv

def setup_project_environment():
    print("=== Preparing Project Environment ===")
    dirs = [
        "/data/nemo-fraud-detection/notebooks/01_Data_Generation/data", 
        "/data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft"
    ]
    
    for d in dirs:
        path = Path(d)
        if path.exists():
            print(f"ℹ️ Directory already exists: {d}/")
        else:
            path.mkdir(parents=True, exist_ok=True)
            print(f"📁 Directory created: {d}/")
            
    print()
    
    # Load .env if present
    env_path = Path(".env")
    if env_path.exists():
        load_dotenv(env_path)
        print("ℹ️ .env file already exists and has been loaded.")
    
    # Check current values from environment variables
    current_hf = os.getenv("HF_TOKEN", "").strip()
    current_ngc = os.getenv("NGC_API_KEY", "").strip()
    current_wandb = os.getenv("WANDB_API_KEY", "").strip()
    
    # Prompt if any key is missing
    needs_input = not current_hf or not current_ngc or not current_wandb
    
    if needs_input:
        print("=== API Key Configuration ===")
        if not current_hf:
            hf_input = input("Enter your Hugging Face Token (optional, press Enter to skip): ").strip()
            if hf_input:
                current_hf = hf_input
                os.environ["HF_TOKEN"] = current_hf
        else:
            print("ℹ️ HF_TOKEN is already set.")
            
        if not current_ngc:
            ngc_input = input("Enter your NGC API Key (optional, press Enter to skip): ").strip()
            if ngc_input:
                current_ngc = ngc_input
                os.environ["NGC_API_KEY"] = current_ngc
        else:
            print("ℹ️ NGC_API_KEY is already set.")

        if not current_wandb:
            wandb_input = input("Enter your Weights & Biases API Key (optional, press Enter to skip): ").strip()
            if wandb_input:
                current_wandb = wandb_input
                os.environ["WANDB_API_KEY"] = current_wandb
        else:
            print("ℹ️ WANDB_API_KEY is already set.")
            
        # Write .env file with complete state (new or updated)
        env_content = []
        if current_hf:
            env_content.append(f"HF_TOKEN={current_hf}")
        if current_ngc:
            env_content.append(f"NGC_API_KEY={current_ngc}")
        if current_wandb:
            env_content.append(f"WANDB_API_KEY={current_wandb}")
            
        if env_content:
            env_path.write_text("\n".join(env_content) + "\n")
            print("✅ .env file successfully updated/saved.")
    else:
        print("ℹ️ All required API keys are already set in the environment.")

setup_project_environment()

=== Preparing Project Environment ===
📁 Directory created: /data/nemo-fraud-detection/notebooks/01_Data_Generation/data/
📁 Directory created: /data/nemo-fraud-detection/notebooks/02_Data_Curation/data/sft/

ℹ️ .env file already exists and has been loaded.
=== API Key Configuration ===
ℹ️ HF_TOKEN is already set.
ℹ️ NGC_API_KEY is already set.


### Step 6: Start Containers & Display Access Guide
Starts the entire stack in the background and automatically determines the server IP as well as the access commands.

In [ ]:
import socket
import subprocess
from pathlib import Path

def get_host_ip():
    try:
        s = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
        s.connect(("8.8.8.8", 80))
        ip = s.getsockname()[0]
        s.close()
        return ip
    except Exception:
        return "localhost"

def start_stack_and_show_guide():
    print("=== 1. Checking Active Docker Containers ===")
    try:
        ps_res = subprocess.run(["docker", "ps"], capture_output=True, text=True, check=True)
        if ps_res.stdout.strip():
            print(ps_res.stdout)
        else:
            print("ℹ️ No active Docker containers found.\n")
    except Exception as e:
        print(f"⚠️ Could not retrieve Docker status: {e}\n")

    print("=== 2. Checking Data Directory ===")
    data_dir = Path("/data/nemo-fraud-detection/notebooks/01_Data_Generation/data")
    if data_dir.exists():
        print(f"✅ Directory already exists: {data_dir}\n")
    else:
        print(f"⚠️ Warning: Directory does not exist yet: {data_dir}\n")

    print("=== 3. Starting Docker Compose Stack ===")
    try:
        res = subprocess.run(["docker", "compose", "up", "-d"], capture_output=True, text=True, check=True)
        print("🚀 All containers started successfully!\n")
    except subprocess.CalledProcessError as e:
        print(f"❌ Error starting the Docker stack:\n{e.stderr}\n")
        return

    server_ip = get_host_ip()

    print("=" * 65)
    print("🌐 ACCESS & CONNECTIVITY GUIDE")
    print("=" * 65)

    print("\n1. 🐳 NeMo Container:")
    print("   - Terminal access (SSH/Host):")
    print("     👉 docker exec -it nemo-24.09 bash")
    print("   - Start Jupyter Notebook inside (if needed):")
    print("     👉 jupyter notebook --ip=0.0.0.0 --port=8889 --no-browser --allow-root")
    print("   - Verification: Open the URL in your browser. The Jupyter interface should load.")

    print("\n2. 🌸 NeMo Automodel Container:")
    print("   - Terminal access (SSH/Host):")
    print("     👉 docker exec -it nemo-automodel bash")
    print("   - Start Jupyter Notebook inside (if needed):")
    print("     👉 jupyter notebook --ip=0.0.0.0 --port=8887 --no-browser --allow-root")
    print(f"   - Web URL: http://{server_ip}:8887 (or http://localhost:8887)")
    print("   - Verification: Open the URL in your browser. The Jupyter interface should load.")

    print("\n3. 📈 Grafana Dashboard (Monitoring):")
    print(f"   - Web URL: http://{server_ip}:3000")
    print("   - Default login: admin / admin")
    print("   - Verification: Check whether the NVIDIA DCGM Exporter dashboard displays data (GPU utilization/VRAM).")

    print("\n=== 3. Current Docker Containers on System ===")
    try:
        # Show all containers (including stopped ones so Prometheus, dcgm-exporter, etc. are visible)
        res = subprocess.run(["docker", "ps", "-a"], capture_output=True, text=True, check=True)
        if res.stdout.strip():
            print(res.stdout)
        else:
            print("ℹ️ No Docker containers found.")
    except Exception as e:
        print(f"⚠️ Could not retrieve Docker status: {e}")

start_stack_and_show_guide()

In [ ]:
import socket
import urllib.request
import json

def check_container_connectivity():
    print("\n=== 4. Network & Connectivity Checks ===")
    
    # Endpoints to test within the Docker network (or via localhost depending on where the script runs)
    # When running directly on the host, we use localhost and the mapped ports:
    services_to_check = [
        ("NeMo Automodel (Jupyter)", "http://localhost:8887"),
        ("NIM Llama 3 LLM", "http://localhost:8800/v1/models"), # Standard OpenAI-compatible endpoint of NIM
        ("Prometheus Monitoring", "http://localhost:9090"),
        ("Grafana Dashboard", "http://localhost:3000"),
        ("DCGM Exporter (GPU)", "http://localhost:9400/metrics")
    ]
    
    for name, url in services_to_check:
        try:
            # Short HTTP request to check whether the service responds
            req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
            with urllib.request.urlopen(req, timeout=3) as response:
                print(f"✅ {name} is reachable (Status: {response.getcode()})")
        except Exception as e:
            print(f"⚠️ {name} is not responding yet (it may still be starting up): {e}")

    # Optional: Test whether the NIM container responds specifically
    print("\n--- Testing Connection to LLM (NIM) ---")
    try:
        url = "http://localhost:8800/v1/models"
        req = urllib.request.Request(url)
        with urllib.request.urlopen(req, timeout=5) as response:
            data = json.loads(response.read().decode())
            print(f"🚀 LLM container reached successfully! Available models: {[m['id'] for m in data.get('data', [])]}")
    except Exception:
        print("ℹ️ NIM container is loading the model in the background (this can take a few minutes on initial startup).")

# Call at the end of setup:
check_container_connectivity()